# noisechisel-torch on GPU vs Gnuastro's NoiseChisel (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PabloMSanAla/noisechisel/blob/cupy-ccl/notebooks/colab_gpu_benchmark.ipynb)

This notebook:
1. checks the CUDA stack (driver, PyTorch, CuPy);
2. installs `noisechisel-torch[cuda]` and Gnuastro 0.24 (`astnoisechisel`);
3. runs the test-suite (including the GPU and bit-identical-to-Gnuastro tests);
4. benchmarks, on three Roman WFI frames from Google Drive, **Gnuastro** vs **noisechisel-torch CPU** vs **noisechisel-torch CUDA**
   and checks that the detection mask, Sky and Sky STD are identical.

**Runtime → Change runtime type → GPU** before running.

## 1. GPU / CUDA check

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
import torch, os
print("torch", torch.__version__, "| CUDA build", torch.version.cuda,
      "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
print("GPU:", torch.cuda.get_device_name(0), "| CPU cores:", os.cpu_count())

## 2. Install noisechisel-torch and a single matching CuPy

Colab may already ship a CuPy build (e.g. `cupy-cuda13x`). Exactly one `cupy-cudaXXx`
must be installed, matching the driver's CUDA major version: mismatched ones are removed.
**If this cell uninstalls anything, restart the session** (Runtime → Restart session)
and continue from section 1.

In [ ]:
import re, subprocess, sys, importlib.metadata as md

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
driver_cuda = int(re.search(r"CUDA Version:\s*(\d+)", smi).group(1))
want = f"cupy-cuda{min(driver_cuda, 13)}x"
have = []
for d in ("cupy", "cupy-cuda11x", "cupy-cuda12x", "cupy-cuda13x"):
    try:
        md.version(d); have.append(d)
    except md.PackageNotFoundError:
        pass
print("driver CUDA", driver_cuda, "| wanted", want, "| installed", have)
removed = [d for d in have if d != want]
if removed:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", *removed], check=True)
if want not in have:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", want], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "noisechisel-torch[fits,test] @ git+https://github.com/PabloMSanAla/noisechisel.git@cupy-ccl"], check=True)
if removed:
    print("\n>>> Removed", removed, "- RESTART THE SESSION (Runtime > Restart session) before continuing.")

In [ ]:
!rm -rf nc_repo && git clone -q -b cupy-ccl https://github.com/PabloMSanAla/noisechisel.git nc_repo

In [ ]:
import cupy, cupyx.scipy.ndimage as cnd
import noisechisel_torch as nct
from noisechisel_torch.core import _cupy_available
print("noisechisel-torch", nct.__version__, "| cupy", cupy.__version__,
      "| CUDA runtime", cupy.cuda.runtime.runtimeGetVersion())
# CuPy smoke test + zero-copy exchange with torch
m = torch.rand(1000, 1000, device="cuda") > 0.6
lab, n = cnd.label(cupy.from_dlpack(m.to(torch.uint8)))
print("cupy label OK:", int(n), "components;", "torch<->cupy DLPack OK:",
      torch.from_dlpack(lab).device)
print("ccl chosen by default on CUDA:", nct.NoiseChisel(device="cuda").ccl)
assert _cupy_available()

## 3. Install Gnuastro 0.24

Ubuntu's `apt` package is too old, so `astnoisechisel` 0.24 comes from conda-forge via
a standalone micromamba in its own prefix (it does not touch Colab's Python).
`libgnuastro` on linux-64 links OpenBLAS directly, hence the `libblas` pin.

In [ ]:
%%bash
set -e
cd /content
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
./bin/micromamba create -y -q -p /content/gnuastro -c conda-forge gnuastro=0.24 "libblas=*=*openblas" > /dev/null
/content/gnuastro/bin/astnoisechisel --version | head -1

In [ ]:
os.environ["PATH"] = "/content/gnuastro/bin:" + os.environ["PATH"]
!astnoisechisel --version | head -1

## 4. Test-suite (CPU, CUDA, CuPy and bit-identical-to-Gnuastro tests)

In [ ]:
!cd nc_repo && python -m pytest -q -rs

## 5. Data

Three Roman WFI frames (4088×4088, float32, HDU 0) downloaded from Google Drive.
Set `USE_SYNTHETIC = True` to also add synthetic Roman-like images
(Sky/σ ≈ 4·10⁴, which stresses float32/float64 arithmetic).

In [ ]:
import gdown
import numpy as np
from astropy.io import fits

GDRIVE = {
    "frame1": "1KVOIrK4gzOQQ_UwNTRicQDWzzD_Vvp-E",
    "frame2": "1EcCdtAWNHgXXBFj5ZigiKDgbSL1yFOGF",
    "frame3": "1Kqu8yRDW_PM_QkBwKFs-G1I1L2l95wib",
}
USE_SYNTHETIC = False
DATA = "/content/frames"; os.makedirs(DATA, exist_ok=True)

FILES = []
for name, fid in GDRIVE.items():
    p = f"{DATA}/{name}.fits"
    if not os.path.exists(p):
        gdown.download(id=fid, output=p, quiet=True)
    with fits.open(p) as hl:
        print(name, f"{os.path.getsize(p)/1e6:.1f} MB", hl[0].data.shape, hl[0].data.dtype)
    FILES.append(p)

def synthetic(n=4088, sky=0.2846, sigma=7.08e-6, nsrc=4000, seed=0):
    rng = np.random.default_rng(seed)
    img = rng.normal(sky, sigma, (n, n))
    for _ in range(int(nsrc * (n / 4088) ** 2)):
        y, x = rng.integers(32, n - 32, 2)
        s = rng.uniform(1.0, 5.0)
        a = rng.lognormal(np.log(10 * sigma), 1.2)
        r = int(5 * s) + 1
        yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
        img[y - r:y + r + 1, x - r:x + r + 1] += a * np.exp(-(yy**2 + xx**2) / (2 * s * s))
    return img.astype(np.float32)

if USE_SYNTHETIC:
    p = f"{DATA}/synthetic_4088.fits"
    if not os.path.exists(p):
        fits.writeto(p, synthetic(), overwrite=True)
    FILES.append(p)
FILES

## 6. Benchmark: Gnuastro vs noisechisel-torch (CPU / CUDA)

In [ ]:
import subprocess, time, pandas as pd
from noisechisel_torch import NoiseChisel, read_image

REPEAT = 3            # timed repetitions (minimum is kept), after one warm-up
TORCH_CCL_GPU = True  # also time the pure-torch labelling on GPU (slower)

def time_gnuastro(path, out):
    cmd = ["astnoisechisel", path, "-h0", f"--output={out}", "--quiet"]
    subprocess.run(cmd, check=True)                       # warm-up (file cache)
    ts = []
    for _ in range(REPEAT):
        t0 = time.perf_counter(); subprocess.run(cmd, check=True); ts.append(time.perf_counter() - t0)
    with fits.open(out) as hl:
        return min(ts), (hl["DETECTIONS"].data.astype(bool), hl["SKY"].data.astype(np.float32),
                         hl["SKY_STD"].data.astype(np.float32))

def time_torch(img, device, ccl):
    nc = NoiseChisel(device=device, ccl=ccl)
    nc(img)                                               # warm-up
    ts = []
    for _ in range(REPEAT):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter(); res = nc(img)
        if device == "cuda": torch.cuda.synchronize()
        ts.append(time.perf_counter() - t0)
    return min(ts), (res.detections.cpu().numpy(), res.sky_image().cpu().numpy(),
                     res.std_image().cpu().numpy())

def ndiff(a, b):
    return [int(np.count_nonzero(x != y)) for x, y in zip(a, b)]

configs = [("cpu", "scipy"), ("cuda", "cupy")] + ([("cuda", "torch")] if TORCH_CCL_GPU else [])
rows = []
for path in FILES:
    img, _ = read_image(path)
    name = os.path.basename(path)
    t_g, ref = time_gnuastro(path, f"{DATA}/{name}_gnuastro.fits")
    rows.append(dict(image=name, program=f"Gnuastro 0.24 ({os.cpu_count()} threads)",
                     time_s=t_g, mask_diff=0, sky_diff=0, std_diff=0))
    print(f"{name}: Gnuastro {t_g:.2f}s", flush=True)
    for dev, ccl in configs:
        t, out = time_torch(img, dev, ccl)
        dm, ds, dd = ndiff(ref, out)
        rows.append(dict(image=name, program=f"torch {dev} ccl={ccl}", time_s=t,
                         mask_diff=dm, sky_diff=ds, std_diff=dd))
        print(f"  torch {dev:4s} ccl={ccl:5s} {t:7.3f}s  speed-up x{t_g / t:5.1f}  "
              f"pixels differing mask/sky/std: {dm}/{ds}/{dd}", flush=True)

df = pd.DataFrame(rows)
df["speedup_vs_gnuastro"] = df.groupby("image")["time_s"].transform(lambda s: s.iloc[0] / s)
df

In [ ]:
import matplotlib.pyplot as plt
piv = df.pivot(index="program", columns="image", values="time_s")
ax = piv.plot.barh(figsize=(8, 3.5), logx=True)
ax.set_xlabel("time per image [s] (log)"); ax.set_ylabel("")
ax.set_title(torch.cuda.get_device_name(0))
plt.tight_layout(); plt.show()

## 7. Where the GPU time goes (per-stage timings)

In [ ]:
img, _ = read_image(FILES[-1])
NoiseChisel(device="cuda", ccl="cupy", verbose=True)(img);

## 8. Summary to paste back

In [ ]:
print(f"GPU {torch.cuda.get_device_name(0)}, {os.cpu_count()} CPU cores, torch {torch.__version__}, cupy {cupy.__version__}")
print(df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))